In [1]:
from pathlib import Path
from datetime import date
import re, json, os

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import HfApi
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

print("Libraries loaded successfully!")
print("Project root:", ROOT)
print("torch", torch.__version__, "| MPS available:", torch.backends.mps.is_available())

Libraries loaded successfully!
Project root: /Users/harris/Documents/government-policy-assistant
torch 2.15.0 | MPS available: True


In [2]:
ontario_df = load_dataset("a2aj/canadian-laws", data_dir="LEGISLATION-ON", split="train").to_pandas()
bc_df      = load_dataset("a2aj/canadian-laws", data_dir="LEGISLATION-BC", split="train").to_pandas()

print("Ontario documents:", len(ontario_df))
print("BC documents:", len(bc_df))

README.md:   0%|          | 0.00/12.8k [00:00<?, ?B/s]

LEGISLATION-ON/train.parquet: reconstructing file:   0%|          |  0.00B / 64.1MB            

LEGISLATION-ON/train.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

LEGISLATION-BC/train.parquet: reconstructing file:   0%|          |  0.00B / 26.6MB            

LEGISLATION-BC/train.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Ontario documents: 856
BC documents: 572


In [3]:
DATASET_SHA = HfApi().dataset_info("a2aj/canadian-laws").sha
print("Dataset revision:", DATASET_SHA)
print("Downloaded:", date.today())

Dataset revision: 7aeeba3ff0ec7edfb24018d48d38e26a19a7bf4d
Downloaded: 2026-11-02


In [4]:
#What columns exist
print("Columns:")
print(ontario_df.columns.tolist())

display(ontario_df[["name_en", "citation_en", "document_date_en", "source_url_en", "upstream_license"]].head())

Columns:
['citation_en', 'citation2_en', 'dataset', 'name_en', 'document_date_en', 'source_url_en', 'scraped_timestamp_en', 'unofficial_text_en', 'unofficial_sections_en', 'num_sections_en', 'citation_fr', 'citation2_fr', 'name_fr', 'document_date_fr', 'source_url_fr', 'scraped_timestamp_fr', 'unofficial_text_fr', 'unofficial_sections_fr', 'num_sections_fr', 'upstream_license']


,name_en,citation_en,document_date_en,source_url_en,upstream_license
0,"Arthur Wishart Act (Franchise Disclosure), 2000","SO 2000, c 3",2020-09-01 00:00:00+00:00,https://www.ontario.ca/laws/api/v2/legislation...,See upstream license: https://perma.cc/5PT4-EX...
1,"Christopher's Law (Sex Offender Registry), 2000","SO 2000, c 1",2026-07-01 00:00:00+00:00,https://www.ontario.ca/laws/api/v2/legislation...,See upstream license: https://perma.cc/5PT4-EX...
2,"City of Kawartha Lakes Act, 2000","SO 2000, c 43",2003-01-01 00:00:00+00:00,https://www.ontario.ca/laws/api/v2/legislation...,See upstream license: https://perma.cc/5PT4-EX...
3,"Deaf-Blind Awareness Month Act, 2000","SO 2000, c 34",2000-12-21 00:00:00+00:00,https://www.ontario.ca/laws/api/v2/legislation...,See upstream license: https://perma.cc/5PT4-EX...
4,"Electronic Commerce Act, 2000","SO 2000, c 17",2020-05-15 00:00:00+00:00,https://www.ontario.ca/laws/api/v2/legislation...,See upstream license: https://perma.cc/5PT4-EX...


In [5]:
#Data types and size
ontario_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 856 entries, 0 to 855
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   citation_en             856 non-null    str                
 1   citation2_en            0 non-null      str                
 2   dataset                 856 non-null    str                
 3   name_en                 856 non-null    str                
 4   document_date_en        856 non-null    datetime64[us, UTC]
 5   source_url_en           856 non-null    str                
 6   scraped_timestamp_en    856 non-null    datetime64[us, UTC]
 7   unofficial_text_en      856 non-null    str                
 8   unofficial_sections_en  856 non-null    str                
 9   num_sections_en         856 non-null    int64              
 10  citation_fr             855 non-null    str                
 11  citation2_fr            0 non-null      str             

In [6]:
#Missing Values:
print("Ontario missing values:")
display(ontario_df.isnull().sum())

print("\nBC missing values:")
display(bc_df.isnull().sum())

Ontario missing values:


citation_en                 0
citation2_en              856
dataset                     0
name_en                     0
document_date_en            0
source_url_en               0
scraped_timestamp_en        0
unofficial_text_en          0
unofficial_sections_en      0
num_sections_en             0
citation_fr                 1
citation2_fr              856
name_fr                     1
document_date_fr            1
source_url_fr               1
scraped_timestamp_fr        1
unofficial_text_fr          1
unofficial_sections_fr      1
num_sections_fr             0
upstream_license            0
dtype: int64


BC missing values:


citation_en                 0
citation2_en              572
dataset                     0
name_en                     0
document_date_en            0
source_url_en               0
scraped_timestamp_en        0
unofficial_text_en          0
unofficial_sections_en      6
num_sections_en             0
citation_fr               572
citation2_fr              572
name_fr                   572
document_date_fr          572
source_url_fr             572
scraped_timestamp_fr      572
unofficial_text_fr        572
unofficial_sections_fr    572
num_sections_fr             0
upstream_license          572
dtype: int64

In [7]:
#Which licesnes apply
print("Ontario licences:")
display(ontario_df["upstream_license"].value_counts())
print("\nBC licences:")
display(bc_df["upstream_license"].value_counts())

Ontario licences:


upstream_license
See upstream license: https://perma.cc/5PT4-EXXN © King's Printer for Ontario. NOTE: This is an unofficial reproduction of Ontario legislation, made without endorsement or affiliation by the Government of Ontario.    856
Name: count, dtype: int64


BC licences:


Series([], Name: count, dtype: int64)

In [8]:
#Look for matches in titles
for name, df in [("Ontario", ontario_df), ("BC", bc_df)]:
    m = df[df["name_en"].str.contains("Employment Standards", case=False, na=False)]
    print(f"\n{name} matches:")
    display(m[["name_en", "citation_en", "document_date_en", "num_sections_en"]])


Ontario matches:


,name_en,citation_en,document_date_en,num_sections_en
5,"Employment Standards Act, 2000","SO 2000, c 41",2026-01-01 00:00:00+00:00,259



BC matches:


,name_en,citation_en,document_date_en,num_sections_en
137,Employment Standards Act,"RSBC 1996, c 113",1996-01-01 00:00:00+00:00,160


In [9]:
#Pick an Act
def pick_act(df, exact_name):
    m = df[df["name_en"].str.strip().str.lower() == exact_name.lower()]
    assert len(m) == 1, f"Expected 1 match for {exact_name!r}, got {len(m)}"
    return m.iloc[0]

ontario_act = pick_act(ontario_df, "Employment Standards Act, 2000")
bc_act      = pick_act(bc_df,      "Employment Standards Act")

for a in (ontario_act, bc_act):
    print(a["name_en"], "|", a["citation_en"], "|", a["document_date_en"])
    print("  sections:", a["num_sections_en"], "| has section text:", pd.notna(a["unofficial_sections_en"]))
    print("  licence:", a["upstream_license"])
    print("  url:", a["source_url_en"])

Employment Standards Act, 2000 | SO 2000, c 41 | 2026-01-01 00:00:00+00:00
  sections: 259 | has section text: True
  licence: See upstream license: https://perma.cc/5PT4-EXXN © King's Printer for Ontario. NOTE: This is an unofficial reproduction of Ontario legislation, made without endorsement or affiliation by the Government of Ontario.
  url: https://www.ontario.ca/laws/api/v2/legislation/en/doc-search/statute/00e41
Employment Standards Act | RSBC 1996, c 113 | 1996-01-01 00:00:00+00:00
  sections: 160 | has section text: True
  licence: nan
  url: https://www.bclaws.gov.bc.ca/civix/document/id/complete/statreg/96113_01/xml


In [10]:
#Save the rows
pd.DataFrame([ontario_act, bc_act]).to_parquet(RAW / "esa_raw_rows.parquet")
print("Saved to", RAW / "esa_raw_rows.parquet")

Saved to /Users/harris/Documents/government-policy-assistant/data/raw/esa_raw_rows.parquet


In [11]:
#When was the Act scrapped 
for a in (ontario_act, bc_act):
    print(a["name_en"], "| scraped:", a["scraped_timestamp_en"])

ontario_sections = json.loads(ontario_act["unofficial_sections_en"])
bc_sections      = json.loads(bc_act["unofficial_sections_en"])

print("\nOntario sections:", len(ontario_sections))
print("BC sections:", len(bc_sections))
print("\nOntario keys:", list(ontario_sections)[:40])
print("\nBC keys:", list(bc_sections)[:40])

Employment Standards Act, 2000 | scraped: 2026-10-31 11:08:20.443000+00:00
Employment Standards Act | scraped: 2026-10-31 11:21:06.975000+00:00

Ontario sections: 259
BC sections: 160

Ontario keys: ['1', '2', '3', '3.1', '4', '5', '5.1', '6', '7', '8', '8.1', '8.2', '8.3', '8.4', '8.5', '8.6', '8.7', '9', '10', '11', '12', '12.1', '13', '14', '14.1', '14.2', '14.3', '14.4', '14.5', '15', '15.1', '16', '17', '17.1', '17.2', '17.3', '18', '19', '20', '21']

BC keys: ['1', '2', '3', '3.1', '4', '5', '6', '7', '8', '9', '9.1', '10', '11', '12', '13', '14', '15', '16', '16.1', '16.2', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '30.1', '30.2', '30.3', '30.4', '31', '32']


In [12]:
#Look at raw text in one section
print(repr(ontario_sections[list(ontario_sections)[0]][:500]))

'(1) In this Act,\n“agent” includes a trade union that represents an employee in collective bargaining; (“mandataire”)\n“alternative vacation entitlement year” means, with respect to an employee, a recurring 12-month period that begins on a date chosen by the employer, other than the first day of the employee’s employment; (“année de référence différente”)\n“arbitrator” includes,\n(a) a board of arbitration, and\n(b) the Board, when it is acting under section 133 of the Labour Relations Act, 1995; (“a'


In [13]:
#Confirm every key is a number
import re
for name, secs in [("Ontario", ontario_sections), ("BC", bc_sections)]:
    odd = [k for k in secs if not re.fullmatch(r"\d+(\.\d+)*", k)]
    print(name, "non-numeric keys:", odd)

Ontario non-numeric keys: []
BC non-numeric keys: []


In [14]:
#Print full sections of text that will be used in Test Set
target_sections = {   # candidates: you'll drop or add some after reading
    "Ontario":          {"vacation_entitlement": ["33", "34", "35"],
                         "vacation_pay": ["35.2", "36"],
                         "overtime": ["22"]},
    "British Columbia": {"vacation_entitlement": ["57", "59"],
                         "vacation_pay": ["58", "60"],
                         "overtime": ["40", "37", "42"]},
}
all_sections = {"Ontario": ontario_sections, "British Columbia": bc_sections}

for prov, topics in target_sections.items():
    for topic, nums in topics.items():
        for n in nums:
            print(f"\n===== {prov} | {topic} | s.{n} =====")
            print(all_sections[prov].get(n, "!!! NOT FOUND"))


===== Ontario | vacation_entitlement | s.33 =====
(1) An employer shall give an employee a vacation of,
(a) at least two weeks after each vacation entitlement year that the employee completes, if the employee’s period of employment is less than five years; or
(b) at least three weeks after each vacation entitlement year that the employee completes, if the employee’s period of employment is five years or more.
(2) Both active employment and inactive employment shall be included for the purposes of subsection (1).
(3) If an employee does not take vacation in complete weeks, the employer shall base the number of days of vacation that the employee is entitled to,
(a) on the number of days in the employee’s regular work week; or
(b) if the employee does not have a regular work week, on the average number of days the employee worked per week during the most recently completed vacation entitlement year.
(4) Clause (1) (b) requires employers to provide employees with a period of employment of

In [15]:
#Load Part map of Act
part_map = json.load(open(PROC / "part_map.json"))

HELD_OUT_PARTS = {
    "Ontario": {"VII", "VIII", "XI"},          # hours of work, overtime, vacation
    "British Columbia": {"4", "7"},            # hours of work & overtime, annual vacation
}
DEFINITION_SECTIONS = {"Ontario": {"1"}, "British Columbia": {"1"}}
TABLE_SECTIONS = {"British Columbia": {"3", "16.2", "45", "49.1", "52.13"}}   # from verification_log.md

target_lookup = {(prov, sec): topic
                 for prov, topics in target_sections.items()
                 for topic, secs in topics.items() for sec in secs}

verification = pd.read_json(ROOT / "results" / "verification_report.json", dtype={"section": str})
verified_lookup = {(r.jurisdiction, r.section): r.status == "match" for r in verification.itertuples()}

In [19]:
#Build one record per section
def split_role(prov, sec, part, text):
    if part in HELD_OUT_PARTS[prov]:
        return "test_only"
    if (sec in DEFINITION_SECTIONS.get(prov, set()) or sec in TABLE_SECTIONS.get(prov, set())
            or part is None or text.strip().lower().lstrip("[").startswith("repealed")):
        return "exclude"
    return "train_pool"

BC_LICENCE = "King's Printer Licence – British Columbia v1.1 (https://www.bclaws.gov.bc.ca/standards/Licence.html); © King's Printer, Victoria, British Columbia, Canada"
records = []
for prov, code, act, sections in [("Ontario", "ON", ontario_act, ontario_sections),
                                  ("British Columbia", "BC", bc_act, bc_sections)]:
    for sec, text in sections.items():
        info = part_map[prov].get(sec, {})
        part = info.get("part")
        records.append({
            "chunk_id": f"{code}-ESA-{sec}",
            "jurisdiction": prov,
            "document": act["name_en"],
            "citation": act["citation_en"],
            "section": sec,
            "part": part,
            "part_title": info.get("part_title"),
            "split_role": split_role(prov, sec, part, text),
            "is_target": (prov, sec) in target_lookup,
            "eval_topic": target_lookup.get((prov, sec)),
            "text": text,
            "character_count": len(text),
            "source_url": act["source_url_en"],
            "scraped_at": str(act["scraped_timestamp_en"]),
            "upstream_license": act["upstream_license"] if pd.notna(act["upstream_license"]) else BC_LICENCE,
            "dataset_revision": DATASET_SHA,
            "verified_against_official": verified_lookup.get((prov, sec), False),
        })

kb_df = pd.DataFrame(records)
assert kb_df["chunk_id"].is_unique
assert not kb_df[kb_df.is_target & (kb_df.split_role != "test_only")].shape[0], "a target section is not test_only!"
print("Total sections:", len(kb_df))

Total sections: 419


In [20]:
#Checking sections
display(kb_df.groupby(["jurisdiction", "split_role"]).size().unstack(fill_value=0))
display(kb_df[kb_df.is_target][["chunk_id", "part", "part_title", "eval_topic", "split_role", "verified_against_official"]])
display(kb_df.sort_values("character_count", ascending=False).head(10)[["chunk_id", "split_role", "character_count"]])

split_role,exclude,test_only,train_pool
jurisdiction,,,
British Columbia,19,18,123
Ontario,9,24,226


,chunk_id,part,part_title,eval_topic,split_role,verified_against_official
44,ON-ESA-22,VIII,Overtime Pay,overtime,test_only,True
59,ON-ESA-33,XI,Vacation With Pay,vacation_entitlement,test_only,True
60,ON-ESA-34,XI,Vacation With Pay,vacation_entitlement,test_only,True
61,ON-ESA-35,XI,Vacation With Pay,vacation_entitlement,test_only,True
63,ON-ESA-35.2,XI,Vacation With Pay,vacation_pay,test_only,True
64,ON-ESA-36,XI,Vacation With Pay,vacation_pay,test_only,True
303,BC-ESA-37,4,Hours of Work and Overtime,overtime,test_only,True
306,BC-ESA-40,4,Hours of Work and Overtime,overtime,test_only,True
308,BC-ESA-42,4,Hours of Work and Overtime,overtime,test_only,True
335,BC-ESA-57,7,Annual Vacation,vacation_entitlement,test_only,True


,chunk_id,split_role,character_count
253,ON-ESA-141,train_pool,21361
96,ON-ESA-50.1,train_pool,13928
0,ON-ESA-1,exclude,13860
88,ON-ESA-49.4,train_pool,8500
259,BC-ESA-1,exclude,8427
97,ON-ESA-50.1.1,train_pool,8103
416,BC-ESA-127,train_pool,7903
29,ON-ESA-15,train_pool,6235
161,ON-ESA-74.11,train_pool,6233
109,ON-ESA-56,train_pool,5727


In [21]:
kb_df.to_json(PROC / "kb_sections.jsonl", orient="records", lines=True, force_ascii=False)
print("Saved", len(kb_df), "sections to", PROC / "kb_sections.jsonl")

Saved 419 sections to /Users/harris/Documents/government-policy-assistant/data/processed/kb_sections.jsonl
